# 🇻🇳 Fine-Tuning DeepSeek-OCR for Vietnamese Handwriting Recognition

This notebook demonstrates the end-to-end pipeline for adapting **DeepSeek-OCR (3.4B Vision-Language Model)** to recognize Vietnamese handwritten text using **PEFT (LoRA)** and **Unsloth** on consumer-grade GPUs (e.g. Kaggle Tesla T4).

## 1. Environment & Dependencies

In [ ]:
%%capture
!pip install unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2
!pip install jiwer einops addict easydict

## 2. Load Model & Tokenizer with Unsloth (16-bit LoRA Mode)

In [ ]:
import os
os.environ["UNSLOTH_WARN_UNINITIALIZED"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
from transformers import AutoModel
from unsloth import FastVisionModel
from huggingface_hub import snapshot_download

# Download weights locally to Kaggle working directory
snapshot_download("unsloth/DeepSeek-OCR", local_dir="/kaggle/working/deepseek_ocr")

# Load base model
model, tokenizer = FastVisionModel.from_pretrained(
    "/kaggle/working/deepseek_ocr",
    load_in_4bit=False,
    auto_model=AutoModel,
    trust_remote_code=True,
    unsloth_force_compile=True,
    use_gradient_checkpointing="unsloth",
)

## 3. Attach LoRA PEFT Adapters (Train only 2.27% parameters)

In [ ]:
model = FastVisionModel.get_peft_model(
    model,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    random_state=3407,
    use_rslora=False,
)

# Enable fast training
FastVisionModel.for_training(model)

## 4. Dataset Preparation & Conversation Formatting

In [ ]:
import json
from PIL import Image

images_path = "/kaggle/input/split-uit-hwdb-line/kaggle/working/split_UIT_HWDB_line/images"
train_path = "/kaggle/input/split-uit-hwdb-line/kaggle/working/split_UIT_HWDB_line/train.json"
val_path = "/kaggle/input/split-uit-hwdb-line/kaggle/working/split_UIT_HWDB_line/validation.json"

with open(train_path, "r", encoding="utf-8") as f:
    train_labels = json.load(f)
with open(val_path, "r", encoding="utf-8") as f:
    val_labels = json.load(f)

instruction = "<image>\nFree OCR. "

def convert_to_conversation(raw_data):
    dataset_samples = []
    for img_name, text in raw_data.items():
        img_file = os.path.join(images_path, img_name)
        img_obj = Image.open(img_file).convert("RGB")
        conversation = [
            {"role": "<|User|>", "content": instruction, "images": [img_obj]},
            {"role": "<|Assistant|>", "content": text},
        ]
        dataset_samples.append({"messages": conversation})
    return dataset_samples

dataset_train = convert_to_conversation(train_labels)
dataset_val = convert_to_conversation(val_labels)
print(f"[*] Loaded {len(dataset_train)} train and {len(dataset_val)} val samples.")

## 5. Custom DataCollator (Gundam Dynamic Cropping + Prompt Masking)

In [ ]:
from src.data.collator import DeepSeekOCRDataCollator

data_collator = DeepSeekOCRDataCollator(
    tokenizer=tokenizer,
    model=model,
    image_size=640,
    base_size=1024,
    crop_mode=True,
    train_on_responses_only=True,
)

## 6. Training with Hugging Face Trainer

In [ ]:
from transformers import Trainer, TrainingArguments

trainer = Trainer(
    model=model,
    tokenizer=tokenizer,
    data_collator=data_collator,
    train_dataset=dataset_train,
    eval_dataset=dataset_val,
    args=TrainingArguments(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        gradient_checkpointing=True,
        warmup_steps=10,
        num_train_epochs=1,
        learning_rate=2e-4,
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.001,
        lr_scheduler_type="linear",
        seed=3407,
        fp16=True,
        output_dir="outputs",
        report_to="none",
        dataloader_num_workers=2,
        remove_unused_columns=False,
        eval_strategy="steps",
        save_strategy="steps",
        eval_steps=25,
        save_steps=25,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
    ),
)

trainer_stats = trainer.train()

## 7. Save Fine-Tuned Checkpoint & Demo Inference

In [ ]:
final_dir = "deepseek_ocr_finetuned_final"
trainer.save_model(final_dir)
tokenizer.save_pretrained(final_dir)

FastVisionModel.for_inference(model)
sample_image = os.path.join(images_path, "100.jpg")
output_text = model.infer(
    tokenizer,
    prompt="<image>\nFree OCR.",
    image_file=sample_image,
    base_size=1024,
    image_size=640,
    crop_mode=True,
)
print("[*] Recognized Text:", output_text)